# Project 2 — CLIP Text-to-Image Search (CIFAR-10)
Search **1,000 real images** using any short English phrase. Change `IMAGES_PER_CLASS` to search more. This uses a **pretrained CLIP** model (inference only; no training). CIFAR-10 has 10 categories and 10,000 test images, each 32×32 pixels.

In [ ]:
!pip -q install transformers pillow matplotlib

## 1. Load CLIP
The first run downloads pretrained weights. GPU is recommended but optional.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

MODEL_NAME = 'openai/clip-vit-base-patch32'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
model = CLIPModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()
processor = CLIPProcessor.from_pretrained(MODEL_NAME)
print('Device:', DEVICE)

## 2. Download CIFAR-10 and select images
CIFAR-10 has 10 classes: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck. The full **test split** has 10,000 images. We select the same number from each class to make a balanced gallery.

In [ ]:
from tensorflow.keras.datasets import cifar10

IMAGES_PER_CLASS = 100  # 100 = 1,000 images; 500 = 5,000; 1000 = all 10,000 test images.
CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

(x_train, y_train), (x_test, y_test) = cifar10.load_data()
labels = y_test.ravel()
rng = np.random.default_rng(42)
selected = np.concatenate([rng.choice(np.where(labels == c)[0], size=min(IMAGES_PER_CLASS, 1000), replace=False) for c in range(10)])
images = [Image.fromarray(x_test[i]) for i in selected]
names = [CLASS_NAMES[labels[i]] for i in selected]
print('Gallery size:', len(images))

# Preview a few random gallery images.
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, i in zip(axes.flat, rng.choice(len(images), size=10, replace=False)):
    ax.imshow(images[i]); ax.set_title(names[i]); ax.axis('off')
plt.tight_layout(); plt.show()

## 3. Encode all gallery images (once)
CLIP converts each image into a 512-dimensional embedding. Process small batches to avoid GPU/CPU memory problems. More images take longer to index, but subsequent text searches are fast.

In [ ]:
BATCH_SIZE = 64
all_vectors = []

with torch.no_grad():
    for start in range(0, len(images), BATCH_SIZE):
        batch_images = images[start:start + BATCH_SIZE]
        inputs = processor(images=batch_images, return_tensors='pt').to(DEVICE)
        vectors = model.get_image_features(**inputs)
        if hasattr(vectors, 'pooler_output'):
            vectors = vectors.pooler_output
        vectors = torch.nn.functional.normalize(vectors, dim=-1)
        all_vectors.append(vectors.cpu())
        if start % (BATCH_SIZE * 5) == 0:
            print(f'Encoded {min(start + BATCH_SIZE, len(images))}/{len(images)} images')

image_vectors = torch.cat(all_vectors)
print('Image embedding matrix:', tuple(image_vectors.shape))

## 4. Search with any English phrase
CLIP encodes your text and ranks the gallery by cosine similarity. **It always returns the nearest images**, even if the gallery contains no true match. Scores are not probabilities.

In [ ]:
def search_images(query, top_k=6):
    with torch.no_grad():
        inputs = processor(text=[query], return_tensors='pt', padding=True, truncation=True).to(DEVICE)
        text_vector = model.get_text_features(**inputs)
        if hasattr(text_vector, 'pooler_output'):
            text_vector = text_vector.pooler_output
        text_vector = torch.nn.functional.normalize(text_vector, dim=-1).cpu()
        scores = (text_vector @ image_vectors.T).squeeze(0).numpy()

    best = np.argsort(-scores)[:min(top_k, len(images))]
    fig, axes = plt.subplots(1, len(best), figsize=(3 * len(best), 3.5))
    for ax, i in zip(np.atleast_1d(axes), best):
        ax.imshow(images[i].resize((160, 160)))
        ax.set_title(f'{names[i]}  {scores[i]:.3f}')
        ax.axis('off')
    plt.suptitle(query); plt.tight_layout(); plt.show()
    return [(names[i], round(float(scores[i]), 3)) for i in best]

search_images('a photo of a dog')
search_images('something that can fly')

## 5. Try your own prompt

In [ ]:
query = input('Describe the image you want to find: ')
search_images(query, top_k=6)

## Student experiments
1. Compare `a dog`, `a puppy`, and `a small furry animal`.
2. Try `a vehicle`, `a flying machine`, and `an animal in nature`.
3. Increase `IMAGES_PER_CLASS` from 100 to 300 or 1000 and rerun Sections 2–5.
4. Search for `a pizza` — there is no pizza class. What does CLIP return?
5. Why are images encoded only once while text is encoded for each search?

**Limitations:** CIFAR-10 images are tiny (32×32), and there are only 10 labeled categories. CLIP rankings may be noisy. No model training occurs in this notebook.